In [30]:

from itertools import product as prdt

def ZZMOD_voltage_cover(X, G, alpha):
    X_edges = list(X.edges())
    X_verts = list(X.vertices())
    Y = DiGraph(loops=True, multiedges=True)
    Y_verts = list(prdt(X_verts, G.list()))
    X_cross_G = list(prdt(X_edges, G.list()))
    Y_edges = []
    for (e, g) in X_cross_G:
        u, v, label = e
        tail = (u, g)
        head = (v, g + alpha(e))
        Y_edges.append((tail, head, label))
    
    Y.add_vertices(Y_verts)
    Y.add_edges(Y_edges)
    return Y

In [ ]:
def cyclotomic_bouquet_graph(p, n):
    X = DiGraph(loops=True, multiedges=True)
    X.add_edge(0, 0, (0, 0)) 
    
    for i in range(1, p**n + 1):        
        for j in range(1, i + 1):
            if i % p != 0:      
                X.add_edge(0, 0, (i, j))
        
    return X

In [77]:
def cyclotomic_voltage_cover(p, n):
    # DELTA = AdditiveAbelianGroup(euler_phi(p**n))
    # G = AdditiveAbelianGroup([euler_phi(p**n)])
    if p == 2:
        ret = DiGraph(loops=True, multiedges=True)
        DELTA = AdditiveAbelianGroup([0])
        
        return ret, DELTA, lambda e: 1
    
    R1ZMOD = ZZ**1
    pn_subgroup = span([[p**n]], ZZ)
    DELTA  = R1ZMOD.quotient(pn_subgroup)
    gen = DELTA.gen(0)
    zeta_p = primitive_root(p**n)
    
    def phi(i):
        if i == 0:
            return DELTA.zero()
        
        k = discrete_log(Mod(i, p**n), Mod(zeta_p, p**n))
        return k * gen
    
    def alpha(e):
        label = e[2]              
        i = label[0]
        if i == 0:
            return phi(1)         
        else:
            return -phi(i) 
     
    X = cyclotomic_bouquet_graph(p, n)
    print(f"{factor(p^n)}th-Cyclotomic Bouquet Graph")
    X.show()
    ret = ZZMOD_voltage_cover(X, DELTA, alpha) 
    print(f"{factor(p^n)}th-Cyclotomic Bouquet Graph Voltage Assignment")
    ret.show()
    return ret, DELTA, alpha





In [23]:
def n_bouquet_graph_family(p, n):
    return {i : cyclotomic_bouquet_graph(p, i) for i in range(1, n+1)}

In [ ]:
import random as rm
def t_integralization(t, p, n):
    if p == 2:
        return []

    phi_p_n = euler_phi(p**n)
    DELTA = AdditiveAbelianGroup([phi_p_n])
    QQ_DELTA = GroupAlgebra(DELTA, QQ)
    gen = DELTA.gen(0)
    zeta_p = primitive_root(p**n)
    
    def phi(i):
        if i == 0:
            return DELTA.zero()
        
        k = discrete_log(Mod(i, p**n), Mod(zeta_p, p**n))
        return k*gen
    
    lhs_sum_lst = []
    for a in range(1, p**n + 1):
        if gcd(a, p) == 1:
            lhs_sum_lst.append((a * QQ_DELTA(-phi(a)))/(p**n))
    lhs_sum_lst = [(QQ_DELTA(phi(t)) + QQ_DELTA(-t))*(QQ_DELTA(elt)) for elt in lhs_sum_lst]
    ret_lst = []
    for i in range(0, phi_p_n):
        ret_lst.append(sum([x.coefficient(DELTA([i])) for x in lhs_sum_lst]))
    return ret_lst

primes = [a for a in range(3, 20) if is_prime(a)]
t = primes[rm.randint(0,len(primes)-1)]
p_sum_theta_coef_dict = {}
old_primes_stack = []
for i in range(1, 3):
    for j, p in enumerate(primes):
        """
        <a>-t<at**(-1)> ≡ 0 mod m
        """    
        if gcd(p, t) == 1:
            t_integralization_theta = t_integralization(t, p, i)
            p_sum_theta_coef = p-sum(t_integralization_theta)
            p_sum_theta_coef_dict[p] = p_sum_theta_coef
            old_primes_stack.append(p_sum_theta_coef)
            # if i==1 and j == 0:
            #     print(f"t == {t}        p == {p}        i == {i}        j == {j}        {t_integralization_theta}        sum(coef. of theta) % t == {p_sum_theta_coef % t}\n")
            # else:
            #     print(f"                p == {p}        i == {i}        j == {j}        {t_integralization_theta}       sum(coef. of theta) % t == {p_sum_theta_coef % t}\n")
        print(f"    Computing <a>-t<at**(-1)> for p == {p}, i == {i}, and t == {t}:")
        for a in range(1, p**i+1):
            if gcd(a,p) == 1:
                print(f"        a == {a}    a/t == {QQ(a)/QQ(t)}    fractional part of a/t == {QQ((a % t))/QQ(t)}   <a>-t<at**(-1)> == {a-t*((a % t)/t)}")
"""
NOTE: 07/21/2026. CHECK TO MAKE SURE <a>-t<at**(-1)> IS CORRECT.
"""

    Computing <a>-t<at**(-1)> for p == 3, i == 1, and t == 11:
        a == 1    a/t == 1/11    fractional part of a/t == 1/11   <a>-t<at**(-1)> == 0.0
        a == 2    a/t == 2/11    fractional part of a/t == 2/11   <a>-t<at**(-1)> == 0.0
    Computing <a>-t<at**(-1)> for p == 5, i == 1, and t == 11:
        a == 1    a/t == 1/11    fractional part of a/t == 1/11   <a>-t<at**(-1)> == 0.0
        a == 2    a/t == 2/11    fractional part of a/t == 2/11   <a>-t<at**(-1)> == 0.0
        a == 3    a/t == 3/11    fractional part of a/t == 3/11   <a>-t<at**(-1)> == 0.0
        a == 4    a/t == 4/11    fractional part of a/t == 4/11   <a>-t<at**(-1)> == 0.0
    Computing <a>-t<at**(-1)> for p == 7, i == 1, and t == 11:
        a == 1    a/t == 1/11    fractional part of a/t == 1/11   <a>-t<at**(-1)> == 0.0
        a == 2    a/t == 2/11    fractional part of a/t == 2/11   <a>-t<at**(-1)> == 0.0
        a == 3    a/t == 3/11    fractional part of a/t == 3/11   <a>-t<at**(-1)> == 0.0
        a 

In [31]:
def generate_nth_Z_p_tower(X, p, n, alpha):
    Z_p_tower_dict = {}
    for i in range(0, n-1):
        Y_i = ZZMOD_voltage_cover(X, AdditiveAbelianGroup(euler_phi(p**n), alpha))
        Z_p_tower_dict[i+1] = (Y_i, Y_i.adjacency_matrix())
    
    return  Z_p_tower_dict

In [ ]:
def cyclotomic_fields_double_towers(X, p, n):
    X = generate_nth_Z_p_tower(X, p, n)
    cyclotomic_VA_dict = {}
    
    for i in range(1, n):
        Yi, DELTA_PN, alpha_pn = cyclotomic_voltage_cover(X[i], p, i)
        def pi_alpha = lambda
        
        # cyclotomic_VA_dict[i] = (, )
    """
    NOTE: Need
    """

    # α: Ex ⟶ Δ_n-->Δ_n/Δ+, Δ+ = unique group of order p
    
    